# ChatBot

## Check which python

In [1]:
import sys
print(sys.executable)

d:\Code\swe_agent_jom\.venv\Scripts\python.exe


## load deepseek api

In [4]:
from pathlib import Path
from dotenv import load_dotenv
import os

env_path = Path.cwd() / ".env"
load_dotenv(dotenv_path=env_path)

deepseek_api_key = os.getenv("DEEPSEEK_API_KEY")
print("DEEPSEEK_API_KEY loaded:", deepseek_api_key is not None)

DEEPSEEK_API_KEY loaded: True


## LLM API Call Example

In [6]:
from openai import OpenAI

client = OpenAI(
    api_key=deepseek_api_key,
    base_url="https://api.deepseek.com",
)

response = client.chat.completions.create(
    model="deepseek-v4-flash",
    messages=[
        {
            "role": "system",
            "content": "You are a research helper. Be accurate and concise"
        },
        {
            "role": "user",
            "content": "Explain what is AI Agent in 3 key points."
        }
    ],
    stream=False,
)

print(response.choices[0].message.content)

Here are three key points about AI Agents:

1. **Autonomous Decision-Making** – An AI agent operates independently, perceiving its environment through sensors (e.g., data, cameras) and taking actions to achieve specific goals without constant human guidance.

2. **Goal-Oriented & Adaptive** – It is designed to pursue a defined objective, using reasoning and learning to adapt its behavior based on feedback from the environment (e.g., rewards, outcomes).

3. **Environment Interaction** – The agent continuously cycles between perceiving, reasoning, and acting within a dynamic environment—whether physical (robots) or digital (software bots, chatbots).


## Stream output & temperature

In [7]:
stream  = client.chat.completions.create(
    model="deepseek-v4-flash",
    messages=[
        {
            "role": "system",
            "content": "You are a rigorous research assistant. Your responses should be well-structured, verifiable, and free from fabricated information."
        },
        {
            "role": "user",
            "content": "Introduce Agent Memory to me with 3 key point."
        }
    ],
    stream=True,
    temperature=0.1 # Smaller lead to more stable and comservative, larger lead to more creative and emanative
)

for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end="")

Here are three key points about **Agent Memory** in the context of AI agents (such as LLM-based assistants):

1. **Two Core Memory Types**  
   Agent memory is typically divided into *short-term* (working context, e.g., the current conversation window) and *long-term* (persistent storage, e.g., embeddings in a vector database). Short-term memory enables real‑time coherence, while long-term memory allows the agent to recall facts, user preferences, or past interactions across sessions.

2. **Essential for Coherence and Personalization**  
   Without memory, an agent treats every interaction as isolated. Memory lets the agent maintain consistent behavior (e.g., remembering that a user prefers formal language), avoid repeating information, and build on previous decisions—critical for tasks like multi‑step planning, dialogue, or user‑specific recommendations.

3. **Implementation via Retrieval-Augmented Systems**  
   In practice, agent memory is often implemented using **retrieval-augment

## Structured output

In [8]:
client = OpenAI(
    api_key=deepseek_api_key,
    base_url="https://api.deepseek.com",
)

response = client.chat.completions.create(
    model="deepseek-v4-flash",
    messages=[
        {"role": "system", "content": "You are a research helper. Be accurate and concise. You can only output valid json object."},
        {"role": "user", "content": "Explain what is AI Agent in 3 key points."}
    ],
    stream=False,
    response_format={"type": "json_object"},
)
print(response.choices[0].message.content)

{
  "points": [
    "An AI agent is a software entity that perceives its environment through sensors and acts upon it through actuators to achieve specific goals.",
    "It operates autonomously, making decisions and taking actions without human intervention, often using machine learning or rule-based systems.",
    "AI agents can be reactive (responding to current inputs) or proactive (planning ahead), and range from simple chatbots to complex robotic systems."
  ]
}


## Multiple rounds of conversation

In [24]:
from openai import OpenAI
from dotenv import load_dotenv, find_dotenv
import os

load_dotenv(find_dotenv())

client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com",
)

MODEL = "deepseek-v4-flash"  

messages = [
    {
        "role": "system",
        "content": "You are a rigorous research assistant. Be consice and accurate."
    }
]

def chat(user_input: str, messages=messages) -> str|None:
    messages.append({
        "role": "user",
        "content": user_input
    })

    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0.2,
    )

    assistant_reply = response.choices[0].message.content

    messages.append({
        "role": "assistant",
        "content": assistant_reply
    })

    return assistant_reply

In [25]:
chat("My name is JOM, I am working on AI Agents.")

"Hello JOM. I understand you're working on AI Agents. How can I assist you today?"

In [26]:
chat("What is my name? And what I am focusing on?")

'Your name is JOM, and you are focusing on AI Agents.'

In [27]:
chat("How do you know that?")

'You stated it in your first message: "My name is JOM, I am working on AI Agents." I simply retrieved that information from the history of our conversation.'

In [28]:
chat("What if the message is too large?")

'If the message (or conversation history) is too large, my ability to retrieve specific information like your name depends on the model\'s context window and retrieval mechanisms. \n\n- **Context window limit**: If the total tokens exceed my maximum context length, older parts of the conversation may be truncated. In that case, I would lose the memory of your earlier statement ("My name is JOM, I am working on AI Agents").\n- **No external memory**: I do not have access to any long-term storage or database beyond the current conversation. Once truncated, the information is gone.\n- **Possible mitigation**: In practical systems, large contexts can be handled via summarization, sliding windows, or vector databases for retrieval-augmented generation (RAG).\n\nWithout any of those mechanisms, if your initial message is removed from context, I would no longer know your name or focus area.'

In [29]:
for message in messages:
    role = message["role"]
    content = message["content"]

    print(f"{role}: {content}")
    print()

system: You are a rigorous research assistant. Be consice and accurate.

user: My name is JOM, I am working on AI Agents.

assistant: Hello JOM. I understand you're working on AI Agents. How can I assist you today?

user: What is my name? And what I am focusing on?

assistant: Your name is JOM, and you are focusing on AI Agents.

user: How do you know that?

assistant: You stated it in your first message: "My name is JOM, I am working on AI Agents." I simply retrieved that information from the history of our conversation.

user: What if the message is too large?

assistant: If the message (or conversation history) is too large, my ability to retrieve specific information like your name depends on the model's context window and retrieval mechanisms. 

- **Context window limit**: If the total tokens exceed my maximum context length, older parts of the conversation may be truncated. In that case, I would lose the memory of your earlier statement ("My name is JOM, I am working on AI Agents